In [ ]:
import hashlib
import pandas as pd
import numpy as np

import config
from data_loader import fetch_sp500_tickers, load_raw_stock_data
from utils import get_quarter_dates
from portfolio import get_portfolio_performance

# =========================
# 설정
# =========================
COMBO_NAME = "Consumer Staples+Health Care+Information Technology+Utilities"

SECTOR_COUNTS = {
    "Information Technology": 6,
    "Health Care": 5,
    "Consumer Staples": 4,
    "Utilities": 2,
}

N_REPEATS_FIXED = 1000
N_REPEATS_RANDOM = 1000
BASE_SEED = 42

OUT_FIXED_RAW   = "fixed_combo_1000rep_raw.csv"
OUT_RANDOM_RAW  = "random_portfolio_1000rep_raw.csv"
OUT_COMPARE_CSV = "fixed_vs_random_compare_by_quarter.csv"
# =========================


def _stable_seed(*parts: str) -> int:
    msg = "|".join(map(str, parts)).encode("utf-8")
    h = hashlib.md5(msg).hexdigest()[:8]
    return int(h, 16)


def load_data():
    sectors_df, all_tickers = fetch_sp500_tickers()
    sector_map = dict(zip(sectors_df["ticker"], sectors_df["GICS Sector"]))

    raw = load_raw_stock_data(all_tickers, config.START_DATE, config.END_DATE)
    if raw.empty:
        raise RuntimeError("[DATA] No price data loaded.")

    prices_all = raw.xs("Close", level=1, axis=1)
    prices_all.index = pd.to_datetime(prices_all.index)

    valid_tickers = prices_all.columns.tolist()
    sector_map = {t: s for t, s in sector_map.items() if t in valid_tickers}

    return sector_map, prices_all


def sample_sector_tickers(test_prices, sector_map, sector_name, n, rng):
    tickers = [t for t in test_prices.columns if sector_map.get(t) == sector_name]
    tickers = [t for t in tickers if not test_prices[t].isna().all()]

    # 정확히 n개 필요. 부족하면 실패(None)
    if len(tickers) < n:
        return None

    return sorted(rng.choice(tickers, size=n, replace=False).tolist())


def sample_random_tickers(test_prices, n_total, rng):
    tickers = [t for t in test_prices.columns if not test_prices[t].isna().all()]
    if len(tickers) < n_total:
        return None
    return sorted(rng.choice(tickers, size=n_total, replace=False).tolist())


def build_fixed_combo_results(sector_map, prices_all):
    # 섹터명 유효성 체크
    all_sectors = set(sector_map.values())
    for s in SECTOR_COUNTS.keys():
        if s not in all_sectors:
            raise ValueError(f"[SECTOR] 섹터명 불일치: '{s}' (sector_map에 없음)")

    all_quarters = pd.period_range(start=config.START_QUARTER, end=config.END_QUARTER, freq="Q")
    rows = []

    for q in all_quarters:
        start_date, end_date = map(pd.to_datetime, get_quarter_dates(q))
        test_prices = prices_all.loc[start_date:end_date]
        if test_prices.empty:
            continue

        for rep in range(1, N_REPEATS_FIXED + 1):
            seed = BASE_SEED + _stable_seed("FIXED", q, COMBO_NAME, rep)
            rng = np.random.default_rng(seed)

            chosen = []
            chosen_by_sector = {}

            ok = True
            for sector, n in SECTOR_COUNTS.items():
                picks = sample_sector_tickers(test_prices, sector_map, sector, n, rng)
                if picks is None:
                    ok = False
                    break
                chosen.extend(picks)
                chosen_by_sector[sector] = ",".join(picks)

            if not ok:
                continue

            perf = get_portfolio_performance(test_prices[chosen])
            perf["quarter"] = str(q)
            perf["rep"] = rep
            perf["portfolio_type"] = COMBO_NAME
            perf["picked_tickers"] = ",".join(chosen)

            for sector, picks_str in chosen_by_sector.items():
                perf[f"tickers_{sector}"] = picks_str

            rows.append(perf)

    return pd.DataFrame(rows)


def build_random_results(sector_map, prices_all, n_total):
    all_quarters = pd.period_range(start=config.START_QUARTER, end=config.END_QUARTER, freq="Q")
    rows = []

    for q in all_quarters:
        start_date, end_date = map(pd.to_datetime, get_quarter_dates(q))
        test_prices = prices_all.loc[start_date:end_date]
        if test_prices.empty:
            continue

        for rep in range(1, N_REPEATS_RANDOM + 1):
            seed = BASE_SEED + _stable_seed("RANDOM", q, f"N={n_total}", rep)
            rng = np.random.default_rng(seed)

            picks = sample_random_tickers(test_prices, n_total, rng)
            if picks is None:
                continue

            perf = get_portfolio_performance(test_prices[picks])
            perf["quarter"] = str(q)
            perf["rep"] = rep
            perf["portfolio_type"] = f"RANDOM_N{n_total}"
            perf["picked_tickers"] = ",".join(picks)

            rows.append(perf)

    return pd.DataFrame(rows)


def calc_top_percent_metrics(fixed_mean, random_series, higher_is_better: bool) -> dict:
    """
    고정 콤보의 '평균' 성과를 랜덤 1000개 분포와 비교해서
    - better_than_pct: 랜덤 중 고정보다 못한 비율(%)
    - top_pct: 상위 몇 %인지(%)  (높을수록 좋으면 '더 큰 값'이 상위, 낮을수록 좋으면 '더 작은 값'이 상위)
    """
    if random_series.empty or np.isnan(fixed_mean):
        return {"better_than_pct": np.nan, "top_pct": np.nan, "random_mean": np.nan, "random_std": np.nan}

    r = random_series.dropna().values
    if r.size == 0:
        return {"better_than_pct": np.nan, "top_pct": np.nan, "random_mean": np.nan, "random_std": np.nan}

    random_mean = float(np.mean(r))
    random_std = float(np.std(r, ddof=1)) if r.size > 1 else 0.0

    if higher_is_better:
        better_than = float(np.mean(r <= fixed_mean)) * 100.0
        top_pct = 100.0 - better_than  # 예: 95% 이기면 top 5%
    else:
        # 낮을수록 좋음(변동성): r >= fixed 가 많을수록 "내가 더 낮다" = 더 좋다
        better_than = float(np.mean(r >= fixed_mean)) * 100.0
        top_pct = 100.0 - better_than  # 예: 95%가 더 크면(내가 더 낮음) top 5%

    return {
        "better_than_pct": better_than,
        "top_pct": top_pct,
        "random_mean": random_mean,
        "random_std": random_std,
    }


def compare_by_quarter(fixed_df, random_df):
    metrics = ["Cumulative_Return", "Volatility"]
    out_rows = []

    for q in sorted(set(fixed_df["quarter"]).intersection(set(random_df["quarter"]))):
        f_q = fixed_df[fixed_df["quarter"] == q]
        r_q = random_df[random_df["quarter"] == q]

        if f_q.empty or r_q.empty:
            continue

        fixed_mean_ret = float(f_q["Cumulative_Return"].mean())
        fixed_mean_vol = float(f_q["Volatility"].mean())
        random_mean_ret = float(r_q["Cumulative_Return"].mean())
        random_mean_vol = float(r_q["Volatility"].mean())

        ret_rank = calc_top_percent_metrics(
            fixed_mean=fixed_mean_ret,
            random_series=r_q["Cumulative_Return"],
            higher_is_better=True,
        )
        vol_rank = calc_top_percent_metrics(
            fixed_mean=fixed_mean_vol,
            random_series=r_q["Volatility"],
            higher_is_better=False,  # 낮을수록 좋음
        )

        out_rows.append({
            "quarter": q,
            "fixed_portfolio": COMBO_NAME,
            "random_portfolio": random_df["portfolio_type"].iloc[0],

            "fixed_mean_cumret": fixed_mean_ret,
            "random_mean_cumret": random_mean_ret,
            "return_better_than_pct": ret_rank["better_than_pct"],
            "return_top_pct": ret_rank["top_pct"],

            "fixed_mean_vol": fixed_mean_vol,
            "random_mean_vol": random_mean_vol,
            "vol_better_than_pct": vol_rank["better_than_pct"],
            "vol_top_pct": vol_rank["top_pct"],

            "n_fixed_samples": int(len(f_q)),
            "n_random_samples": int(len(r_q)),
        })

    return pd.DataFrame(out_rows).sort_values("quarter").reset_index(drop=True)


def run():
    sector_map, prices_all = load_data()

    n_total = sum(SECTOR_COUNTS.values())

    fixed_df = build_fixed_combo_results(sector_map, prices_all)
    if fixed_df.empty:
        print("[FIXED] 결과가 없습니다. (섹터별 종목 수 부족/데이터 누락 가능)")
        return
    fixed_df.to_csv(OUT_FIXED_RAW, index=False)
    print(f"Saved fixed raw: {OUT_FIXED_RAW}  (rows={len(fixed_df)})")

    random_df = build_random_results(sector_map, prices_all, n_total=n_total)
    if random_df.empty:
        print("[RANDOM] 결과가 없습니다. (분기별 유효 티커 수 부족/데이터 누락 가능)")
        return
    random_df.to_csv(OUT_RANDOM_RAW, index=False)
    print(f"Saved random raw: {OUT_RANDOM_RAW} (rows={len(random_df)})")

    compare_df = compare_by_quarter(fixed_df, random_df)
    compare_df.to_csv(OUT_COMPARE_CSV, index=False)
    print(f"Saved compare: {OUT_COMPARE_CSV}")
    print(compare_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


if __name__ == "__main__":
    run()
